# 07 — Compound optimization (planner inside MetaOrchestrator)

**Same machinery, more compound surface.**

Notebook 06 evolved the prompt of one isolated agent. This notebook evolves the *planner* inside `MetaOrchestrator` and watches the entire orchestration improve downstream — the planner's task decompositions get cleaner, which means each spawned sub-agent gets a sharper subtask, which means the end-to-end success rate climbs.

Score function operates one layer up: we compare the planner's emitted `TaskDecomposition` against an expected shape. The downstream effect is then visible in the optional final cell that runs full orchestrations before vs after.

**Prerequisites:** same as notebook 06. Cost: ~$2–$3.

## 1. Load config + define the planner

In [ ]:
from typing import Any

from pydantic import BaseModel

from orqest import load_config
from orqest.agents import BaseAgent, GlobalState
from orqest.autonomy.meta import TaskDecomposition

config = load_config()
print(f"Model: {config.llm_model}")

INITIAL_PLANNER_PROMPT = (
    "You decompose a high-level goal into 2-4 concrete subtasks. "
    "Each subtask needs a short snake_case name, a one-sentence description, "
    "and requires_agent=True. Keep the list minimal."
)

class PlannerAgent(BaseAgent[GlobalState, TaskDecomposition]):
    async def _run_implementation(self, state: GlobalState, **kwargs: Any) -> TaskDecomposition:
        latest = state.get_latest_message("user") or ""
        result = await self.call_model(latest, state)
        return result.output

## 2. The 15-example gold set

Three buckets:
- **Single-step (5)** — goal needs *one* specialist; planner should not over-decompose.
- **Multi-step (5)** — goal needs 3+ subtasks; planner should produce a coherent ordering.
- **Ambiguous (5)** — goal underspecified; planner should pick a conservative minimal decomposition rather than fabricating scope.

We score by comparing subtask *count* against an expected range plus a name-quality heuristic — not exact-match (LLMs vary on naming) but tight enough to track structural improvements.

In [ ]:
from orqest.optimization import GoldExample

class PlannerExpected(BaseModel):
    min_subtasks: int
    max_subtasks: int
    must_mention: list[str] = []   # snake_case keywords any subtask name should contain at least one of

def _ex(goal: str, lo: int, hi: int, mention: list[str], bucket: str) -> GoldExample:
    return GoldExample[str, PlannerExpected](
        input=goal,
        expected=PlannerExpected(min_subtasks=lo, max_subtasks=hi, must_mention=mention),
        id=f"{bucket}-{hash(goal) % 10000}",
    )

EXAMPLES = [
    # Single-step (1-2 subtasks)
    _ex("Translate this sentence to French: 'The cat sat on the mat.'", 1, 2, ["translat"], "single"),
    _ex("What is 13 * 47?", 1, 2, ["calc", "comput", "answer"], "single"),
    _ex("Define the word 'antidisestablishmentarianism'.", 1, 2, ["defin", "explain"], "single"),
    _ex("Sort this list alphabetically: zebra, apple, mango, banana.", 1, 2, ["sort"], "single"),
    _ex("What's the capital of France?", 1, 2, ["answer", "lookup", "capital"], "single"),
    # Multi-step (3-4 subtasks)
    _ex("Research the latest electric vehicle market, summarize the top 3 manufacturers, and recommend one for a family of 5.", 3, 4, ["research", "summar", "recommend"], "multi"),
    _ex("Build a meal plan: gather dietary requirements, propose a 7-day menu, then list the shopping items.", 3, 4, ["requir", "menu", "shop"], "multi"),
    _ex("Plan a 3-day trip to Tokyo: research attractions, build a daily itinerary, and estimate the total budget.", 3, 4, ["research", "itinerary", "budget"], "multi"),
    _ex("Diagnose a slow API: gather logs, identify the bottleneck, propose a fix, and validate against test cases.", 3, 4, ["log", "bottleneck", "fix", "valid"], "multi"),
    _ex("Design a CLI for converting CSV to JSON: name subcommands, list global flags, write a usage example.", 3, 4, ["subcommand", "flag", "example"], "multi"),
    # Ambiguous (2-3 subtasks — conservative)
    _ex("Help me with my project.", 2, 3, ["clarif", "requir", "scope"], "ambiguous"),
    _ex("Make this better.", 2, 3, ["clarif", "context", "requir"], "ambiguous"),
    _ex("I need some advice.", 2, 3, ["clarif", "context", "topic"], "ambiguous"),
    _ex("Tell me about it.", 2, 3, ["clarif", "context", "subject"], "ambiguous"),
    _ex("Fix the issue.", 2, 3, ["clarif", "identif", "context"], "ambiguous"),
]
print(f"Loaded {len(EXAMPLES)} planner gold examples.")

## 3. Wrap the planner in an Evaluator

Score function checks subtask count is in `[min, max]` and that at least one expected keyword appears across subtask names.

In [ ]:
from orqest.optimization import Evaluator

def make_planner(decoded: dict[str, Any]) -> PlannerAgent:
    return PlannerAgent(
        agent_name="planner",
        system_prompt=decoded["planner.system_prompt"],
        output_type=TaskDecomposition,
        model=config.llm_model,
        api_key=config.llm_api_key,
    )

def score(output: TaskDecomposition, ex: GoldExample[str, PlannerExpected]) -> float:
    expected = ex.expected
    n = len(output.subtasks)
    count_ok = expected.min_subtasks <= n <= expected.max_subtasks

    name_blob = " ".join(s.name.lower() for s in output.subtasks)
    mention_ok = (not expected.must_mention) or any(
        keyword in name_blob for keyword in expected.must_mention
    )

    if count_ok and mention_ok:
        return 1.0
    if count_ok or mention_ok:
        return 0.5
    return 0.0

evaluator = Evaluator(agent_factory=make_planner, score_fn=score)
print("Evaluator ready.")

## 4. Baseline — what does the unoptimized planner do?

In [ ]:
import statistics
from collections import defaultdict

async def measure(decoded: dict[str, Any]) -> dict[str, float]:
    bundles = await evaluator.evaluate_batch(decoded, EXAMPLES)
    by_bucket: dict[str, list[float]] = defaultdict(list)
    for ex, b in zip(EXAMPLES, bundles, strict=True):
        bucket = ex.id.split("-")[0]
        by_bucket[bucket].append(b.accuracy)
    return {bucket: statistics.mean(scores) for bucket, scores in by_bucket.items()} | {
        "overall": statistics.mean(b.accuracy for b in bundles)
    }

baseline = await measure({"planner.system_prompt": INITIAL_PLANNER_PROMPT})
print("Baseline (planner structural quality):")
for k, v in baseline.items():
    print(f"  {k:14s} {v:.3f}")

## 5. Define the genome — one prompt slot, layered constraints

In [ ]:
from orqest.optimization import Genome, PromptGene

genome = Genome(genes=[
    PromptGene(
        name="planner.system_prompt",
        initial=INITIAL_PLANNER_PROMPT,
        constraints=(
            "Match decomposition depth to goal complexity: 1-2 subtasks for trivial goals, "
            "3-4 for genuinely multi-step goals, 2-3 for ambiguous goals (where the first "
            "subtask should typically be clarifying scope rather than fabricating it)."
        ),
    ),
])

## 6. Run the optimizer

In [ ]:
from orqest.optimization import OptimizationConfig, OptimizationRunner

opt_config = OptimizationConfig(
    max_metric_calls=80,
    reflection_model=config.llm_model,
    task_model=config.llm_model,
    minibatch_size=3,
    valset_fraction=0.4,
    seed=42,
)

runner = OptimizationRunner(opt_config, genome=genome, evaluator=evaluator)
result = await runner.optimize(EXAMPLES)
print(f"Best aggregate score: {result.best_score:.3f}")
print(f"Pareto frontier size: {len(result.pareto_candidates)}")

## 7. Diff + per-bucket lift

In [ ]:
from orqest.optimization import apply_result

baseline_planner = make_planner({"planner.system_prompt": INITIAL_PLANNER_PROMPT})
for d in apply_result(result, target=baseline_planner):
    if d.changed:
        print(d.unified)

evolved = await measure(result.best_decoded)
print("\nPer-bucket structural quality:")
print(f"  {'bucket':<12s} {'before':>10s} {'after':>10s} {'delta':>10s}")
for bucket in ("single", "multi", "ambiguous", "overall"):
    b, a = baseline.get(bucket, 0.0), evolved.get(bucket, 0.0)
    arrow = "↑" if a > b else ("↓" if a < b else "=")
    print(f"  {bucket:<12s} {b:>10.3f} {a:>10.3f}     {a - b:+.3f} {arrow}")

## 8. Downstream effect — does the orchestration actually get better?

The planner score is upstream of orchestration success. We've improved the structural metric; let's see whether the cleaner decompositions translate to a better end-to-end run on a held-out goal.

Honest framing: planner improvement directly drives a structural metric. The downstream effect is noisier — many other factors (sub-agent prompts, tool quality, model variance) shape the final result. Don't expect dramatic single-goal lifts; do expect cleaner subtask graphs and fewer retries.

In [ ]:
from orqest.autonomy import AgentFactory, MetaOrchestrator, ToolRegistry

registry = ToolRegistry()
factory = AgentFactory(registry, default_model=config.llm_model, api_key=config.llm_api_key)

HELD_OUT_GOAL = (
    "Plan a small science fair project for an 8th-grader: pick a topic, "
    "design the experiment, list the materials, write a one-paragraph abstract."
)

async def run_orchestration(planner_prompt: str) -> dict[str, Any]:
    planner = make_planner({"planner.system_prompt": planner_prompt})
    meta = MetaOrchestrator(planner, factory, registry, max_subtasks=5)
    res = await meta.solve(HELD_OUT_GOAL)
    return {
        "success": res.success,
        "n_subtasks": len(res.subtask_results),
        "successful_subtasks": sum(1 for r in res.subtask_results if r.success),
        "duration_ms": res.total_duration_ms,
    }

before = await run_orchestration(INITIAL_PLANNER_PROMPT)
after = await run_orchestration(result.best_decoded["planner.system_prompt"])
print(f"BEFORE: {before}")
print(f"AFTER:  {after}")

## What's next

- **Compose with healing** — feed `metacognition.confidence` into `MetricBundle.confidence` and let the optimizer evolve a planner that's both more accurate AND more confident in its decompositions. The plumbing is already there; just pass `confidence_protocol=StructuredOutputProtocol()` to the `Evaluator`.
- **W3 — topology IR.** Once `TopologySpec` ships, the optimizer can mutate not just *what* the planner says but *how the orchestrator wires* the spawned agents (parallel vs sequential, with vs without `RefinementLoop` wrapping). Same `MetricBundle` Pareto contract, much larger search space.
- See `docs/concepts/optimization.md` for the full architecture.